# GSM8K rigorous residual screen

This notebook extends `gsm8k_2pl_irt.ipynb` with more defensible residual diagnostics for contamination screening. It does not replace the perturbation stage; it makes the screen itself harder to fool and easier to calibrate.

The main additions are:

- ability-cross-fitted residuals, so a suspicious answer does not help explain itself through the model ability estimate;
- parametric-bootstrap calibration, so extreme-cell and recurring-item counts can be compared with a fitted no-contamination null;
- posterior-predictive checks for global model fit;
- model-family aggregation, so many related checkpoints do not count like many independent confirmations;
- threshold-sensitivity analysis for the hard-item cutoff and residual cutoff.

The notebook uses the saved files produced by the original notebook. Expensive sections are controlled by flags in the configuration cell and are off by default.

## Interpretation rule

A large residual is an anomaly, not a contamination proof. The strongest claim this notebook can support is: under the fitted IRT baseline and calibration checks, these model-item successes are unusually surprising and deserve perturbation testing.

A contamination claim still requires a follow-up design where suspect models solve the original item but fail difficulty-matched perturbation twins more often than appropriate controls.

In [ ]:
# Configuration
from pathlib import Path

BASE_DIR = Path('.')
MATRIX_PARQUET = BASE_DIR / 'gsm8k_matrix_filtered.parquet'
ITEM_PARAMS_CSV = BASE_DIR / 'gsm8k_2pl_item_params.csv'
ABILITIES_CSV = BASE_DIR / 'gsm8k_2pl_abilities.csv'
FLAGGED_CELLS_CSV = BASE_DIR / 'gsm8k_flagged_cells.csv'
OUTPUT_DIR = BASE_DIR / 'rigorous_residual_outputs'

# Conservative defaults: only cheap sections run unless you opt in.
RUN_BASELINE = True
RUN_CROSS_FIT = False
RUN_BOOTSTRAP = False
RUN_POSTERIOR_PREDICTIVE = False
RUN_SENSITIVITY = False

# Set this True for a fast end-to-end check on a small slice.
SMOKE_MODE = False
SMOKE_N_MODELS = 80
SMOKE_N_ITEMS = 120
SMOKE_BOOTSTRAPS = 3

RANDOM_SEED = 20260701
N_FOLDS = 5
B_BOOT = 200
B_PPC = 200
HARD_Q = 0.75
Z_FLAG = 4.0
ITEM_RECURRENCE_MIN_MODELS = 3
ITEM_RECURRENCE_MIN_FAMILIES = 3

# Ability scoring settings.
THETA_PRIOR_SD = 2.5
THETA_BOUNDS = (-6.0, 6.0)
THETA_MAX_ITER = 35
THETA_TOL = 1e-4

OUTPUT_DIR.mkdir(exist_ok=True)
print('configured output directory:', OUTPUT_DIR.resolve())

In [ ]:
# Imports
import json
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit

warnings.filterwarnings('ignore')
rng = np.random.default_rng(RANDOM_SEED)
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 160)

## Load and align the saved IRT fit

This section loads the filtered response matrix and the 2PL item/model parameters already produced by the original notebook. Rows are models, columns are GSM8K item ids, and entries are binary correctness values with possible missingness.

In [ ]:
def load_saved_fit():
    matrix = pd.read_parquet(MATRIX_PARQUET)
    if 'model' in matrix.columns:
        matrix = matrix.set_index('model')

    matrix.index = matrix.index.astype(str)
    matrix.columns = [int(c) for c in matrix.columns]
    matrix = matrix.sort_index(axis=1)

    params = pd.read_csv(ITEM_PARAMS_CSV)
    params['item'] = params['item'].astype(int)
    params = params.drop_duplicates('item').set_index('item').sort_index()

    abilities = pd.read_csv(ABILITIES_CSV)
    abilities['model'] = abilities['model'].astype(str)
    abilities = abilities.drop_duplicates('model').set_index('model')

    common_items = [c for c in matrix.columns if c in params.index]
    common_models = [m for m in matrix.index if m in abilities.index]
    matrix = matrix.loc[common_models, common_items]
    params = params.loc[common_items]
    abilities = abilities.loc[common_models]

    return matrix, params, abilities

Mf, params, abilities = load_saved_fit()

if SMOKE_MODE:
    Mf = Mf.iloc[:SMOKE_N_MODELS, :SMOKE_N_ITEMS]
    params = params.loc[Mf.columns]
    abilities = abilities.loc[Mf.index]

U = Mf.to_numpy(dtype=float)
obs_mask = ~np.isnan(U)
items = np.asarray(Mf.columns, dtype=int)
models = np.asarray(Mf.index, dtype=object)
a = params['discrimination'].to_numpy(dtype=float)
b_raw = params['difficulty'].to_numpy(dtype=float)
theta_full = abilities['ability'].to_numpy(dtype=float)

print('matrix:', Mf.shape[0], 'models x', Mf.shape[1], 'items')
print('observed fraction:', round(float(obs_mask.mean()), 4))
print('item params:', params.shape)
print('abilities:', abilities.shape)

## Core residual utilities

The original notebook auto-detected the sign convention for difficulty. We keep that here so the residual code works even if a future IRT library exports difficulty with the opposite sign.

In [ ]:
def predict_2pl(theta, discrimination, difficulty_eff):
    theta = np.asarray(theta, dtype=float)
    discrimination = np.asarray(discrimination, dtype=float)
    difficulty_eff = np.asarray(difficulty_eff, dtype=float)
    return expit(discrimination[None, :] * (theta[:, None] - difficulty_eff[None, :]))


def detect_difficulty_sign(U, theta, discrimination, difficulty_raw):
    observed_item = np.nanmean(U, axis=0)
    candidates = []
    for sign in [1.0, -1.0]:
        P = predict_2pl(theta, discrimination, sign * difficulty_raw)
        predicted_item = np.nanmean(np.where(~np.isnan(U), P, np.nan), axis=0)
        corr = np.corrcoef(observed_item, predicted_item)[0, 1]
        candidates.append((corr, sign))
    corr, sign = max(candidates, key=lambda x: x[0])
    return sign, corr


def standardized_residuals(U, P):
    P = np.clip(P, 1e-6, 1 - 1e-6)
    Z = (U - P) / np.sqrt(P * (1 - P))
    Z[np.isnan(U)] = np.nan
    return Z


def screen_residuals(Z, difficulty_eff, theta, hard_q=HARD_Q, z_flag=Z_FLAG, plausible_rule='median'):
    hard = difficulty_eff > np.nanquantile(difficulty_eff, hard_q)
    if plausible_rule == 'median':
        plausible = theta >= np.nanmedian(theta)
    elif plausible_rule == 'all':
        plausible = np.ones_like(theta, dtype=bool)
    else:
        plausible = theta >= float(plausible_rule)

    eligible = hard[None, :] & plausible[:, None] & ~np.isnan(Z)
    flag_mask = (Z >= z_flag) & eligible
    z_pos = np.where((Z > 0) & eligible, np.minimum(Z, z_flag), 0.0)
    model_score = z_pos.sum(axis=1)
    return flag_mask, model_score, hard, plausible


def flags_to_frame(flag_mask, Z, P, difficulty_eff, models, items):
    ri, ci = np.where(flag_mask)
    return pd.DataFrame({
        'model': models[ri],
        'item': items[ci],
        'z': Z[ri, ci],
        'predicted_p': P[ri, ci],
        'difficulty': difficulty_eff[ci],
    }).sort_values(['z', 'predicted_p'], ascending=[False, True]).reset_index(drop=True)


sign, calibration_corr = detect_difficulty_sign(U, theta_full, a, b_raw)
b_eff = sign * b_raw
print('difficulty convention: P = sigmoid(a * (theta - b_eff)); sign =', sign)
print('item calibration corr:', round(float(calibration_corr), 4))

## Baseline residual screen

This reproduces the residual logic from the original notebook using the saved IRT fit. It is kept as a baseline so the more rigorous diagnostics can be compared against the original screen.

In [ ]:
if RUN_BASELINE:
    P_full = predict_2pl(theta_full, a, b_eff)
    Z_full = standardized_residuals(U, P_full)
    baseline_mask, baseline_score, baseline_hard, baseline_plausible = screen_residuals(Z_full, b_eff, theta_full)
    baseline_flags = flags_to_frame(baseline_mask, Z_full, P_full, b_eff, models, items)
    baseline_screen = pd.DataFrame({
        'model': models,
        'preknowledge_score': baseline_score,
        'ability': theta_full,
        'plausible': baseline_plausible,
    }).sort_values('preknowledge_score', ascending=False).reset_index(drop=True)

    item_recurrence = baseline_flags.groupby('item').agg(
        n_models=('model', 'nunique'),
        median_predicted_p=('predicted_p', 'median'),
        median_z=('z', 'median'),
        difficulty=('difficulty', 'median'),
    ).sort_values(['n_models', 'median_z'], ascending=[False, False])

    print('baseline flagged cells:', len(baseline_flags))
    print('baseline flagged models:', baseline_flags['model'].nunique())
    print('baseline flagged items:', baseline_flags['item'].nunique())
    print('items flagged by >=', ITEM_RECURRENCE_MIN_MODELS, 'models:', int((item_recurrence['n_models'] >= ITEM_RECURRENCE_MIN_MODELS).sum()))
    display(item_recurrence.head(15))

    baseline_flags.to_csv(OUTPUT_DIR / 'baseline_flags.csv', index=False)
    baseline_screen.to_csv(OUTPUT_DIR / 'baseline_model_scores.csv', index=False)
    item_recurrence.to_csv(OUTPUT_DIR / 'baseline_item_recurrence.csv')
else:
    print('baseline screen skipped')

## Ability-cross-fitted residuals

The original residual uses abilities fitted on all item responses, including the target item. Here, for each held-out fold of items, the model ability `theta_m` is re-estimated using only the other items. The held-out residuals are then computed from those held-out predictions.

This is not a full item-parameter cross-fit because the item parameters still come from the saved full 2PL fit. It is nevertheless a useful and much cheaper correction: the suspicious answer no longer directly inflates the model's ability estimate for the same residual.

In [ ]:
def estimate_thetas_newton(Y, discrimination, difficulty_eff, init=None, prior_sd=THETA_PRIOR_SD,
                           bounds=THETA_BOUNDS, max_iter=THETA_MAX_ITER, tol=THETA_TOL):
    Y = np.asarray(Y, dtype=float)
    discrimination = np.asarray(discrimination, dtype=float)
    difficulty_eff = np.asarray(difficulty_eff, dtype=float)
    obs = ~np.isnan(Y)
    Y0 = np.nan_to_num(Y, nan=0.0)
    n_models = Y.shape[0]

    if init is None:
        theta = np.zeros(n_models, dtype=float)
    else:
        theta = np.asarray(init, dtype=float).copy()
        theta = np.where(np.isfinite(theta), theta, 0.0)

    prior_prec = 0.0 if prior_sd is None else 1.0 / (prior_sd ** 2)
    a2 = discrimination ** 2

    for _ in range(max_iter):
        eta = discrimination[None, :] * (theta[:, None] - difficulty_eff[None, :])
        P = expit(np.clip(eta, -35, 35))
        grad = np.sum(np.where(obs, (Y0 - P) * discrimination[None, :], 0.0), axis=1) - prior_prec * theta
        hess = -np.sum(np.where(obs, a2[None, :] * P * (1 - P), 0.0), axis=1) - prior_prec
        step = np.divide(grad, hess, out=np.zeros_like(grad), where=np.abs(hess) > 1e-12)
        step = np.clip(step, -1.0, 1.0)
        theta_new = np.clip(theta - step, bounds[0], bounds[1])
        if np.nanmax(np.abs(theta_new - theta)) < tol:
            theta = theta_new
            break
        theta = theta_new
    return theta


def make_item_folds(n_items, n_folds=N_FOLDS, random_seed=RANDOM_SEED):
    local_rng = np.random.default_rng(random_seed)
    order = local_rng.permutation(n_items)
    folds = np.array_split(order, n_folds)
    return [np.asarray(f, dtype=int) for f in folds]


def ability_crossfit_predictions(U, discrimination, difficulty_eff, n_folds=N_FOLDS, init_theta=None):
    n_models, n_items = U.shape
    folds = make_item_folds(n_items, n_folds=n_folds)
    P_cf = np.full((n_models, n_items), np.nan, dtype=float)
    theta_by_fold = []
    all_idx = np.arange(n_items)

    for fold_id, test_idx in enumerate(folds):
        train_idx = np.setdiff1d(all_idx, test_idx, assume_unique=False)
        theta_train = estimate_thetas_newton(
            U[:, train_idx],
            discrimination[train_idx],
            difficulty_eff[train_idx],
            init=init_theta,
        )
        P_cf[:, test_idx] = predict_2pl(theta_train, discrimination[test_idx], difficulty_eff[test_idx])
        theta_by_fold.append(pd.DataFrame({
            'fold': fold_id,
            'model': models,
            'theta_train_items_only': theta_train,
            'n_test_items': len(test_idx),
        }))
        print('finished fold', fold_id + 1, 'of', n_folds, 'held-out items:', len(test_idx))

    return P_cf, pd.concat(theta_by_fold, ignore_index=True)


print('theta solver smoke check:')
tiny_theta = estimate_thetas_newton(U[:min(5, U.shape[0]), :min(20, U.shape[1])], a[:min(20, len(a))], b_eff[:min(20, len(b_eff))])
print(np.round(tiny_theta, 3))

In [ ]:
if RUN_CROSS_FIT:
    P_cf, theta_by_fold = ability_crossfit_predictions(U, a, b_eff, n_folds=N_FOLDS, init_theta=theta_full)
    Z_cf = standardized_residuals(U, P_cf)

    # Use the original full-fit theta only for the plausibility filter. The residual probabilities are cross-fitted.
    cf_mask, cf_score, cf_hard, cf_plausible = screen_residuals(Z_cf, b_eff, theta_full)
    cf_flags = flags_to_frame(cf_mask, Z_cf, P_cf, b_eff, models, items)
    cf_screen = pd.DataFrame({
        'model': models,
        'crossfit_preknowledge_score': cf_score,
        'full_fit_ability': theta_full,
        'plausible': cf_plausible,
    }).sort_values('crossfit_preknowledge_score', ascending=False).reset_index(drop=True)

    cf_item_recurrence = cf_flags.groupby('item').agg(
        n_models=('model', 'nunique'),
        median_predicted_p=('predicted_p', 'median'),
        median_z=('z', 'median'),
        difficulty=('difficulty', 'median'),
    ).sort_values(['n_models', 'median_z'], ascending=[False, False])

    print('cross-fit flagged cells:', len(cf_flags))
    print('cross-fit flagged models:', cf_flags['model'].nunique())
    print('cross-fit flagged items:', cf_flags['item'].nunique())
    print('items flagged by >=', ITEM_RECURRENCE_MIN_MODELS, 'models:', int((cf_item_recurrence['n_models'] >= ITEM_RECURRENCE_MIN_MODELS).sum()))
    display(cf_item_recurrence.head(15))

    cf_flags.to_csv(OUTPUT_DIR / 'crossfit_flags.csv', index=False)
    cf_screen.to_csv(OUTPUT_DIR / 'crossfit_model_scores.csv', index=False)
    cf_item_recurrence.to_csv(OUTPUT_DIR / 'crossfit_item_recurrence.csv')
    theta_by_fold.to_csv(OUTPUT_DIR / 'crossfit_theta_by_fold.csv', index=False)
else:
    print('cross-fitted residuals skipped; set RUN_CROSS_FIT = True to run')

## Bootstrap calibration

This is the key calibration layer. We simulate response matrices from the fitted 2PL model under a no-contamination null, preserve the original missingness mask, rerun the same screen, and compare observed counts with the null distribution.

The default implementation re-estimates model abilities with fixed item parameters for every bootstrap sample. That makes the null closer to the actual residual workflow without requiring a full py-irt refit hundreds of times.

In [ ]:
def summarize_flag_mask(flag_mask, models, items, min_models=ITEM_RECURRENCE_MIN_MODELS):
    ri, ci = np.where(flag_mask)
    if len(ri) == 0:
        return {
            'flagged_cells': 0,
            'flagged_models': 0,
            'flagged_items': 0,
            'recurring_items': 0,
            'max_models_per_item': 0,
        }
    tmp = pd.DataFrame({'model': models[ri], 'item': items[ci]})
    item_counts = tmp.groupby('item')['model'].nunique()
    return {
        'flagged_cells': int(len(tmp)),
        'flagged_models': int(tmp['model'].nunique()),
        'flagged_items': int(tmp['item'].nunique()),
        'recurring_items': int((item_counts >= min_models).sum()),
        'max_models_per_item': int(item_counts.max()),
    }


def simulate_response_matrix(P, obs_mask, rng):
    S = rng.binomial(1, np.clip(P, 1e-6, 1 - 1e-6)).astype(float)
    S[~obs_mask] = np.nan
    return S


def bootstrap_null_counts(P_null, obs_mask, discrimination, difficulty_eff, theta_init, B=B_BOOT,
                          reestimate_theta=True, use_crossfit=False, n_folds=N_FOLDS):
    rows = []
    for boot in range(B):
        S = simulate_response_matrix(P_null, obs_mask, rng)
        if use_crossfit:
            P_b, _ = ability_crossfit_predictions(S, discrimination, difficulty_eff, n_folds=n_folds, init_theta=theta_init)
            theta_b = theta_init
        elif reestimate_theta:
            theta_b = estimate_thetas_newton(S, discrimination, difficulty_eff, init=theta_init)
            P_b = predict_2pl(theta_b, discrimination, difficulty_eff)
        else:
            theta_b = theta_init
            P_b = P_null

        Z_b = standardized_residuals(S, P_b)
        mask_b, score_b, _, _ = screen_residuals(Z_b, difficulty_eff, theta_b)
        row = summarize_flag_mask(mask_b, models, items)
        row['bootstrap'] = boot
        row['max_model_score'] = float(np.nanmax(score_b))
        rows.append(row)
        if (boot + 1) % max(1, B // 10) == 0:
            print('bootstrap', boot + 1, 'of', B)
    return pd.DataFrame(rows)


def empirical_pvalue(null_values, observed, greater=True):
    null_values = np.asarray(null_values, dtype=float)
    if greater:
        return (1.0 + np.sum(null_values >= observed)) / (len(null_values) + 1.0)
    return (1.0 + np.sum(null_values <= observed)) / (len(null_values) + 1.0)

In [ ]:
if RUN_BOOTSTRAP:
    P_null = predict_2pl(theta_full, a, b_eff)
    B = SMOKE_BOOTSTRAPS if SMOKE_MODE else B_BOOT
    boot_counts = bootstrap_null_counts(
        P_null,
        obs_mask,
        a,
        b_eff,
        theta_full,
        B=B,
        reestimate_theta=True,
        use_crossfit=False,
    )

    observed_mask, observed_score, _, _ = screen_residuals(standardized_residuals(U, P_null), b_eff, theta_full)
    observed_counts = summarize_flag_mask(observed_mask, models, items)
    observed_counts['max_model_score'] = float(np.nanmax(observed_score))

    calibration = []
    for key, observed in observed_counts.items():
        if key in boot_counts.columns:
            calibration.append({
                'statistic': key,
                'observed': observed,
                'null_mean': float(boot_counts[key].mean()),
                'null_p95': float(boot_counts[key].quantile(0.95)),
                'empirical_p_ge_observed': empirical_pvalue(boot_counts[key], observed),
            })
    calibration = pd.DataFrame(calibration)
    display(calibration)

    boot_counts.to_csv(OUTPUT_DIR / 'bootstrap_null_counts.csv', index=False)
    calibration.to_csv(OUTPUT_DIR / 'bootstrap_calibration.csv', index=False)
else:
    print('bootstrap calibration skipped; set RUN_BOOTSTRAP = True to run')

## Posterior-predictive-style checks

This section asks whether the fitted 2PL model can reproduce broad features of the observed response matrix. These are not full Bayesian posterior predictive checks because the saved fit is treated as fixed. They are still valuable parametric predictive checks.

In [ ]:
def matrix_summary(U_like):
    item_pass = np.nanmean(U_like, axis=0)
    model_acc = np.nanmean(U_like, axis=1)
    return {
        'mean_accuracy': float(np.nanmean(U_like)),
        'item_pass_p05': float(np.nanquantile(item_pass, 0.05)),
        'item_pass_p50': float(np.nanquantile(item_pass, 0.50)),
        'item_pass_p95': float(np.nanquantile(item_pass, 0.95)),
        'model_acc_p05': float(np.nanquantile(model_acc, 0.05)),
        'model_acc_p50': float(np.nanquantile(model_acc, 0.50)),
        'model_acc_p95': float(np.nanquantile(model_acc, 0.95)),
    }


def predictive_check(P, obs_mask, B=B_PPC):
    rows = []
    for rep in range(B):
        S = simulate_response_matrix(P, obs_mask, rng)
        row = matrix_summary(S)
        row['replicate'] = rep
        rows.append(row)
    return pd.DataFrame(rows)


def plot_predictive_check(ppc, observed_summary):
    keys = [k for k in observed_summary.keys() if k in ppc.columns]
    fig, axes = plt.subplots(len(keys), 1, figsize=(7, 2.0 * len(keys)))
    if len(keys) == 1:
        axes = [axes]
    for ax, key in zip(axes, keys):
        ax.hist(ppc[key], bins=30, alpha=0.75)
        ax.axvline(observed_summary[key], color='red', linestyle='--', label='observed')
        ax.set_title(key)
        ax.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if RUN_POSTERIOR_PREDICTIVE:
    P_null = predict_2pl(theta_full, a, b_eff)
    B = SMOKE_BOOTSTRAPS if SMOKE_MODE else B_PPC
    observed_summary = matrix_summary(U)
    ppc = predictive_check(P_null, obs_mask, B=B)

    check = []
    for key, obs in observed_summary.items():
        check.append({
            'statistic': key,
            'observed': obs,
            'null_mean': float(ppc[key].mean()),
            'null_p05': float(ppc[key].quantile(0.05)),
            'null_p95': float(ppc[key].quantile(0.95)),
            'empirical_p_low': empirical_pvalue(ppc[key], obs, greater=False),
            'empirical_p_high': empirical_pvalue(ppc[key], obs, greater=True),
        })
    check = pd.DataFrame(check)
    display(check)
    plot_predictive_check(ppc, observed_summary)

    ppc.to_csv(OUTPUT_DIR / 'predictive_check_replicates.csv', index=False)
    check.to_csv(OUTPUT_DIR / 'predictive_check_summary.csv', index=False)
else:
    print('predictive checks skipped; set RUN_POSTERIOR_PREDICTIVE = True to run')

## Model-family aggregation

Many rows are related checkpoints, fine-tunes, merges, or quantizations. This section builds a conservative family key from model names and reports recurrence by distinct families as well as distinct models.

The automatic grouping is intentionally simple. For a paper-quality analysis, replace `infer_model_family` with a curated mapping.

In [ ]:
def infer_model_family(model_name):
    s = str(model_name)
    if '/' in s:
        org, name = s.split('/', 1)
    else:
        org, name = 'unknown', s
    name_l = name.lower()
    name_l = re.sub(r'\b(q[0-9]+|gguf|awq|gptq|int[0-9]+|fp16|bf16)\b', '', name_l)
    name_l = re.sub(r'\b(dpo|sft|rlhf|instruct|chat|preview|beta|alpha|v[0-9]+|it)\b', '', name_l)
    name_l = re.sub(r'[^a-z0-9]+', '-', name_l).strip('-')
    parts = [p for p in name_l.split('-') if p]
    if not parts:
        family = name.lower()
    else:
        family = '-'.join(parts[:3])
    return f'{org.lower()}/{family}'


def add_family_summary(flags):
    out = flags.copy()
    out['family'] = out['model'].map(infer_model_family)
    item_family = out.groupby('item').agg(
        n_models=('model', 'nunique'),
        n_families=('family', 'nunique'),
        median_predicted_p=('predicted_p', 'median'),
        median_z=('z', 'median'),
        difficulty=('difficulty', 'median'),
        example_families=('family', lambda x: ', '.join(sorted(set(x))[:6])),
    ).sort_values(['n_families', 'n_models', 'median_z'], ascending=[False, False, False])
    return out, item_family


if RUN_BASELINE:
    baseline_flags_with_family, baseline_family_recurrence = add_family_summary(baseline_flags)
    print('items flagged by >=', ITEM_RECURRENCE_MIN_FAMILIES, 'families:', int((baseline_family_recurrence['n_families'] >= ITEM_RECURRENCE_MIN_FAMILIES).sum()))
    display(baseline_family_recurrence.head(20))
    baseline_flags_with_family.to_csv(OUTPUT_DIR / 'baseline_flags_with_family.csv', index=False)
    baseline_family_recurrence.to_csv(OUTPUT_DIR / 'baseline_item_family_recurrence.csv')
else:
    print('run the baseline screen first to aggregate family recurrence')

## Threshold sensitivity

The original `Z >= 4` and top-quartile-hard thresholds are reasonable but arbitrary. A robust screen should not depend too sharply on exactly those cutoffs.

In [ ]:
def threshold_sensitivity(U, P, difficulty_eff, theta, hard_grid=(0.70, 0.75, 0.80, 0.85), z_grid=(3.5, 4.0, 4.5, 5.0)):
    Z = standardized_residuals(U, P)
    rows = []
    for hard_q in hard_grid:
        for z_flag in z_grid:
            mask, score, _, _ = screen_residuals(Z, difficulty_eff, theta, hard_q=hard_q, z_flag=z_flag)
            row = summarize_flag_mask(mask, models, items)
            row.update({'hard_q': hard_q, 'z_flag': z_flag, 'max_model_score': float(np.nanmax(score))})
            rows.append(row)
    return pd.DataFrame(rows).sort_values(['hard_q', 'z_flag']).reset_index(drop=True)


if RUN_SENSITIVITY:
    P_full = predict_2pl(theta_full, a, b_eff)
    sensitivity = threshold_sensitivity(U, P_full, b_eff, theta_full)
    display(sensitivity)
    sensitivity.to_csv(OUTPUT_DIR / 'threshold_sensitivity.csv', index=False)
else:
    print('threshold sensitivity skipped; set RUN_SENSITIVITY = True to run')

## Candidate handoff for perturbation

This final section prepares a cleaner candidate list for perturbation. Prefer items that satisfy multiple criteria:

- high recurrence across independent model families;
- low median predicted probability;
- persistence under cross-fitted residuals, if that section has been run;
- stability across thresholds;
- successful prompt/gold-answer mapping.

The output is still a candidate list, not a finding.

In [ ]:
if RUN_BASELINE:
    candidate_items = baseline_family_recurrence.reset_index().copy()
    candidate_items = candidate_items[
        (candidate_items['n_models'] >= ITEM_RECURRENCE_MIN_MODELS)
        & (candidate_items['n_families'] >= ITEM_RECURRENCE_MIN_FAMILIES)
    ].sort_values(['n_families', 'n_models', 'median_predicted_p'], ascending=[False, False, True])

    if Path('gsm8k_flagged_with_gold.csv').exists():
        gold = pd.read_csv('gsm8k_flagged_with_gold.csv')
        gold['item'] = gold['item'].astype(int)
        candidate_items = candidate_items.merge(gold, on='item', how='left')

    print('perturbation candidates:', len(candidate_items))
    display(candidate_items.head(25))
    candidate_items.to_csv(OUTPUT_DIR / 'perturbation_candidate_items.csv', index=False)
else:
    print('run baseline/family sections first to prepare perturbation candidates')

## Reporting checklist

When reporting results from this notebook, include:

1. The exact residual definition and whether it was baseline or ability-cross-fitted.
2. The hard-item cutoff and residual cutoff.
3. The bootstrap null distribution for flagged cells, flagged items, and recurring items.
4. Item recurrence by distinct model families, not only raw model rows.
5. Predictive-check results showing whether the 2PL model reproduces aggregate response patterns.
6. A statement that flagged items are perturbation candidates, not proven contamination.

The mathematically strongest version of the screen is: cross-fitted residuals, bootstrap-calibrated thresholds, family-level recurrence, and perturbation confirmation.